## OPTIMAL EXECUTION WITH ALPHA

Almgren-Chriss liquidation with a directional view: an earnings event whose
alpha arrives in a single instant, solved in closed form with a Green's
function, with and without the event's own jump risk, and compared with the
round-trip version in Paleologo (2021).

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Third-party imports
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Inputs

The Almgren-Chriss Table 1 stock: one million shares at \$50 with daily
volatility $\sigma = 0.95$ \$/share, sold over $T = 5$ days, with temporary
impact $\eta$. Risk aversion is $\lambda = 10^{-6}$, the tangent point of their
efficient frontier, which gives a liquidation rate
$\kappa = \sqrt{\lambda\sigma^2/\eta}$. The earnings print lands mid-horizon
at $t_e$ and pays $a$ dollars on every share held through it ($a = \pm 1$, i.e.
$\pm 2\%$ of the price).

In [3]:
price = 50.0  # $/share
X = 1e6  # shares
T = 5.0  # trading horizon, days
sigma = 0.95  # $/share/sqrt(day)
eta = 2.5e-6  # temporary impact coefficient, $*day/share^2
lam = 1e-6  # risk aversion, 1/$
kappa = np.sqrt(lam * sigma**2 / eta)  # liquidation rate, 1/day

t_e = 2.5  # event time, days
a = 1.0  # event alpha, $/share held through the print

t = np.linspace(0.0, T, 1001)
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

print(
    f"kappa: {kappa:.3f}/day, 1/kappa: {1 / kappa:.2f} days, kappa*T: {kappa * T:.2f}"
)
print(f"impulse a/(2 eta): {a / (2 * eta):,.0f} shares/day")

kappa: 0.601/day, 1/kappa: 1.66 days, kappa*T: 3.00
impulse a/(2 eta): 200,000 shares/day


## Closed forms

The optimal holdings are the liquidation curve plus the Green's function
response to the event,

$$x(t) = X\frac{\sinh\kappa(T-t)}{\sinh\kappa T} + \Big(\frac{a}{2\eta} - \rho\,x_e\Big)G(t, t_e),
\qquad G(t,s) = \frac{\sinh(\kappa t_<)\sinh\kappa(T-t_>)}{\kappa\sinh\kappa T},$$

where $\rho = \lambda\sigma_J^2/\eta$ prices the event's jump variance
$\sigma_J^2$ and $x_e = x(t_e)$ solves

$$x_e = \frac{x_\text{base}(t_e) + G_{ee}\,a/2\eta}{1 + \rho G_{ee}}.$$

With $\rho = 0$ (no event risk) the impulse is just $a/2\eta$. The selling
rate $-\dot x$ follows by differentiating each piece.

In [4]:
def green(t, s, kappa, T):
    """Response at time t to a unit impulse at time s, zero at both ends."""
    t_lo = np.minimum(t, s)
    t_hi = np.maximum(t, s)
    return (
        np.sinh(kappa * t_lo)
        * np.sinh(kappa * (T - t_hi))
        / (kappa * np.sinh(kappa * T))
    )


def green_rate(t, s, kappa, T):
    """Derivative of green in t, taking the right-hand branch at t == s."""
    before = np.cosh(kappa * t) * np.sinh(kappa * (T - s))
    after = -np.sinh(kappa * s) * np.cosh(kappa * (T - t))
    return np.where(t < s, before, after) / np.sinh(kappa * T)


def event_impulse(X, T, kappa, eta, a, t_e, rho=0.0):
    """Size of the impulse at the event, a/(2 eta) - rho x_e."""
    base = X * np.sinh(kappa * (T - t_e)) / np.sinh(kappa * T)
    g_ee = green(t_e, t_e, kappa, T)
    x_e = (base + g_ee * a / (2 * eta)) / (1 + rho * g_ee)
    return a / (2 * eta) - rho * x_e


def holdings(t, X, T, kappa, eta, a, t_e, rho=0.0):
    """Optimal holdings x(t) with an event at t_e."""
    impulse = event_impulse(X, T, kappa, eta, a, t_e, rho)
    base = X * np.sinh(kappa * (T - t)) / np.sinh(kappa * T)
    return base + impulse * green(t, t_e, kappa, T)


def selling_rate(t, X, T, kappa, eta, a, t_e, rho=0.0):
    """Optimal selling rate -dx/dt, in shares per day."""
    impulse = event_impulse(X, T, kappa, eta, a, t_e, rho)
    base = X * kappa * np.cosh(kappa * (T - t)) / np.sinh(kappa * T)
    return base - impulse * green_rate(t, t_e, kappa, T)

## Check against the discrete problem

The discrete objective
$\sum_k \big[\frac{\eta}{\tau}(x_{k-1}-x_k)^2 + \lambda\sigma^2\tau x_k^2 - \alpha_k\tau x_k\big]
+ \lambda\sigma_J^2 x_e^2$
has one first-order condition per interior holding, each linking only
$x_{k-1}$, $x_k$ and $x_{k+1}$: a tridiagonal linear system. The event is a
single bucket $k_e$ with $\alpha_{k_e}\tau = a$. Multiplying each condition by
$\tau/2\eta$,

$$(2 + \kappa^2\tau^2)x_k - x_{k-1} - x_{k+1} + [k = k_e]\big(\rho\tau x_k - a\tau/2\eta\big) = 0.$$

Solving it and comparing with the closed form on the grid, the gap should
shrink like $\tau^2$ (the discrete decay rate $\theta$ differs from $\kappa$ by
$O(\tau^2)$).

In [5]:
def discrete_holdings(X, T, N, kappa, eta, a, k_e, rho=0.0):
    """Solve the discrete first-order conditions for x_0, ..., x_N."""
    tau = T / N
    n = N - 1
    system = (
        np.diag(np.full(n, 2 + (kappa * tau) ** 2)) - np.eye(n, k=1) - np.eye(n, k=-1)
    )
    rhs = np.zeros(n)
    rhs[0] = X
    system[k_e - 1, k_e - 1] += rho * tau
    rhs[k_e - 1] += a * tau / (2 * eta)
    return np.concatenate([[X], np.linalg.solve(system, rhs), [0.0]])


rho_check = lam * (0.05 * price) ** 2 / eta
for N in [50, 200, 1000]:
    grid = np.linspace(0.0, T, N + 1)
    k_e = round(t_e / T * N)
    errors = [
        np.max(
            np.abs(
                discrete_holdings(X, T, N, kappa, eta, a, k_e, rho)
                - holdings(grid, X, T, kappa, eta, a, t_e, rho)
            )
        )
        for rho in [0.0, rho_check]
    ]
    print(
        f"N = {N:4d}: max gap {errors[0]:10.3f} shares (no event risk), "
        f"{errors[1]:10.3f} shares (event risk)"
    )

N =   50: max gap     31.338 shares (no event risk),     42.805 shares (event risk)
N =  200: max gap      1.961 shares (no event risk),      2.677 shares (event risk)
N = 1000: max gap      0.078 shares (no event risk),      0.107 shares (event risk)


## The event trajectory

Holdings and selling rate with no alpha, a positive event ($a = +1$) and a
negative one ($a = -1$). The position is continuous, but the selling rate jumps
by $a/2\eta$ at the print: with good news the trader holds extra inventory into
it and sells faster afterward; with bad news the trader sells ahead of it.

In [6]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Shares Held", "Selling Rate (shares/day)"),
)
for color, (name, alpha) in zip(
    colors, {"No alpha": 0.0, "a = +1": a, "a = -1": -a}.items()
):
    before = t < t_e
    after = ~before
    fig.add_trace(
        go.Scatter(
            x=t,
            y=holdings(t, X, T, kappa, eta, alpha, t_e),
            mode="lines",
            name=name,
            legendgroup=name,
            line={"width": 2, "color": color},
        ),
        row=1,
        col=1,
    )
    # draw the rate on each side of the event separately so the jump shows
    rate = selling_rate(t, X, T, kappa, eta, alpha, t_e)
    for side in [before, after]:
        fig.add_trace(
            go.Scatter(
                x=t[side],
                y=rate[side],
                mode="lines",
                name=name,
                legendgroup=name,
                showlegend=False,
                line={"width": 2, "color": color},
            ),
            row=2,
            col=1,
        )
fig.add_vline(x=t_e, line={"width": 1, "dash": "dot", "color": "gray"})
fig.update_xaxes(title_text="Time (days)", row=2, col=1)
fig.update_layout(
    title="Liquidation Through an Earnings Event",
    width=700,
    height=600,
    margin={"l": 60, "r": 20, "t": 70, "b": 50},
    legend={"x": 1, "xanchor": "right", "y": 1, "yanchor": "top"},
)
fig.write_image("output/event_trajectory.png", scale=2)
fig.show()

rate_jump = selling_rate(t_e, X, T, kappa, eta, a, t_e) - selling_rate(
    t_e - 1e-9, X, T, kappa, eta, a, t_e
)
print(
    f"selling-rate jump at t_e: {rate_jump:,.0f} shares/day (a/2eta = {a / (2 * eta):,.0f})"
)

selling-rate jump at t_e: 200,000 shares/day (a/2eta = 200,000)


## Event timing

The event's contribution alone, $(a/2\eta)\,G(t, t_e)$, for an event early,
mid-horizon and late. The bump reaches only about $1/\kappa$ either side of
the print, and it shrinks near either end of the horizon, where $G$ vanishes:
there is no time to build the extra position before an early event, or to
unwind it after a late one.

In [7]:
fig = go.Figure()
for color, event_time in zip(colors, [0.5, 2.5, 4.5]):
    fig.add_trace(
        go.Scatter(
            x=t,
            y=a / (2 * eta) * green(t, event_time, kappa, T),
            mode="lines",
            name=f"t_e = {event_time} days",
            line={"width": 2, "color": color},
        )
    )
fig.update_layout(
    title="Extra Shares Held Because of the Event",
    xaxis_title="Time (days)",
    yaxis_title="Shares",
    width=700,
    height=400,
    margin={"l": 60, "r": 20, "t": 50, "b": 50},
    legend={"x": 1, "xanchor": "right", "y": 1, "yanchor": "top"},
)
fig.write_image("output/event_timing.png", scale=2)
fig.show()

## Event risk

The print also carries jump variance $\sigma_J^2$, priced as
$\lambda\sigma_J^2 x_e^2$. The position at the print is then a blend,
$x_e = w\,x_\text{base}(t_e) + (1-w)\,x^*$ with $w = 1/(1+\rho G_{ee})$, of where
the liquidation alone would be and the event-only position
$x^* = a/2\lambda\sigma_J^2$. As the jump grows the event wins, and even with
positive alpha the trader can arrive lighter than with no view at all.

In [8]:
x_base_e = X * np.sinh(kappa * (T - t_e)) / np.sinh(kappa * T)
g_ee = green(t_e, t_e, kappa, T)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=t,
        y=holdings(t, X, T, kappa, eta, 0.0, t_e),
        mode="lines",
        name="No alpha",
        line={"width": 2, "color": "gray", "dash": "dash"},
    )
)
print(f"x_base(t_e) = {x_base_e:,.0f} shares, G_ee = {g_ee:.3f} days")
for color, jump_pct in zip(colors, [0.0, 0.01, 0.025, 0.05]):
    sigma_j = jump_pct * price  # $/share
    rho = lam * sigma_j**2 / eta
    x_e = holdings(t_e, X, T, kappa, eta, a, t_e, rho)
    fig.add_trace(
        go.Scatter(
            x=t,
            y=holdings(t, X, T, kappa, eta, a, t_e, rho),
            mode="lines",
            name=f"σ_J = {jump_pct:.1%} of price",
            line={"width": 2, "color": color},
        )
    )
    w = 1 / (1 + rho * g_ee)
    x_star = a / (2 * lam * sigma_j**2) if sigma_j > 0 else np.inf
    print(
        f"sigma_J = {jump_pct:5.1%}: rho = {rho:6.3f}/day, w = {w:.3f}, "
        f"x* = {x_star:,.0f}, x_e = {x_e:,.0f} shares"
    )
fig.add_vline(x=t_e, line={"width": 1, "dash": "dot", "color": "gray"})
fig.update_layout(
    title="Holdings Into the Print, by Jump Risk (a = +1)",
    xaxis_title="Time (days)",
    yaxis_title="Shares Held",
    width=700,
    height=400,
    margin={"l": 60, "r": 20, "t": 50, "b": 50},
    legend={"x": 1, "xanchor": "right", "y": 1, "yanchor": "top"},
)
fig.write_image("output/event_risk.png", scale=2)
fig.show()

x_base(t_e) = 212,148 shares, G_ee = 0.754 days
sigma_J =  0.0%: rho =  0.000/day, w = 1.000, x* = inf, x_e = 362,859 shares
sigma_J =  1.0%: rho =  0.100/day, w = 0.930, x* = 2,000,000, x_e = 337,432 shares
sigma_J =  2.5%: rho =  0.625/day, w = 0.680, x* = 320,000, x_e = 246,680 shares
sigma_J =  5.0%: rho =  2.500/day, w = 0.347, x* = 80,000, x_e = 125,823 shares


## The round trip

Paleologo's version starts and ends flat ($X = 0$): build a position to capture
the event and unwind it. Only the Green's term survives. Three versions: the
finite horizon; the long-horizon limit
$x(t) = \frac{a}{2\eta\kappa}\sinh(\kappa t_<)e^{-\kappa t_>}$, which never
quite unwinds by $T$; and the risk-neutral limit $\lambda \to 0$, where
$G \to t_<(T-t_>)/T$ and the path is a tent, continuous at the print.

In [9]:
t_lo = np.minimum(t, t_e)
t_hi = np.maximum(t, t_e)
round_trips = {
    "Finite horizon": a / (2 * eta) * green(t, t_e, kappa, T),
    "Long horizon (T → ∞)": a
    / (2 * eta * kappa)
    * np.sinh(kappa * t_lo)
    * np.exp(-kappa * t_hi),
    "Risk neutral (λ → 0)": a / (2 * eta) * t_lo * (T - t_hi) / T,
}

fig = go.Figure()
for color, (name, path) in zip(colors, round_trips.items()):
    fig.add_trace(
        go.Scatter(
            x=t,
            y=path,
            mode="lines",
            name=name,
            line={"width": 2, "color": color},
        )
    )
fig.add_vline(x=t_e, line={"width": 1, "dash": "dot", "color": "gray"})
fig.update_layout(
    title="Round Trip Around an Earnings Event (X = 0, a = +1)",
    xaxis_title="Time (days)",
    yaxis_title="Shares Held",
    width=700,
    height=400,
    margin={"l": 60, "r": 20, "t": 50, "b": 50},
    legend={"x": 1, "xanchor": "right", "y": 1, "yanchor": "top"},
)
fig.write_image("output/round_trip.png", scale=2)
fig.show()

# long-horizon buying rate just before and selling rate just after the print
decay = np.exp(-2 * kappa * t_e)
rate_before = a / (4 * eta) * (1 + decay)
rate_after = a / (4 * eta) * (1 - decay)
print(
    f"long horizon: buy {rate_before:,.0f}/day before, sell {rate_after:,.0f}/day after"
)
print(
    f"difference: {rate_before + rate_after:,.0f} shares/day (a/2eta = {a / (2 * eta):,.0f})"
)

long horizon: buy 104,958/day before, sell 95,042/day after
difference: 200,000 shares/day (a/2eta = 200,000)
